In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
!unzip Dataset.zip

!pip install git+https://github.com/speechbrain/speechbrain.git@develop

In [ ]:
from sklearn.model_selection import train_test_split
import torchaudio
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from pathlib import Path
import time
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Used for the summary of the different strategies
# --- DO NOT RERUN BETWEEN DIFFERENT RUNS ---
test_metrics = {}

## Parameters

In [ ]:
num_epochs = 20
batch_size = 4
backbone_lr = 1e-5
classifier_lr = 1e-4

# names of the 3 different fine-tuning strategies
# "frozen": The whole backbone is frozen and only the classifier is trained
# "late_unfreeze": starting with frozen backbone and unfreezing after the 5th (by default) epoch
# "partial_unfreeze": the last 4 (by default) layers are not frozen
strategy = "partial_unfreeze"

output_dir = "results"

In [ ]:
def combine_data(train, dev):
    """ I don't have access to the test set's labels, so I'm combining the train
    and dev sets and later splitting them."""
    name = "train"
    audio_paths = []
    labels = []
    for file in [train, dev]:
        with open(file, 'r') as f:
            lines = [line.strip() for line in f]

            del lines[0] # remove header


            # Mapping food names to labels
            food_label_map = {
                "Apple": 0,
                "Banana": 1,
                "Biscuit": 2,
                "Crisp": 3,
                "Haribo": 4,
                "Nectarine": 5,
                "No_Food": 6
            }

            for line in lines:
                audio_paths.append(f'wav/{name}/{line.split('\t')[0]}')
                food_name = line.split('\t')[4]
                labels.append(food_label_map.get(food_name, -1))
            if -1 in labels:
                raise Exception("Label not found.")

        name = "dev"

    return audio_paths, labels

combined_paths, combined_labels = combine_data('lab/ComParE2015_Eating_train.tsv', 'lab/ComParE2015_Eating_dev.tsv')

train_paths, dev_paths, train_labels, dev_labels = train_test_split(combined_paths, combined_labels, train_size=0.7, random_state=42, stratify=combined_labels)
test_paths, dev_paths, test_labels, dev_labels = train_test_split(dev_paths, dev_labels, train_size=0.5, random_state=42, stratify=dev_labels)

In [ ]:
class AudioDataset(Dataset):
    def __init__(self, paths, labels=None):
        self.paths = paths
        self.labels = labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        waveform, sr = torchaudio.load(self.paths[idx])
        if self.labels is not None:
            return waveform, self.labels[idx]
        else:
            return waveform, idx  # Return index for test set

train = AudioDataset(train_paths, train_labels)
dev = AudioDataset(dev_paths, dev_labels)
test = AudioDataset(test_paths, test_labels)

In [ ]:
def collate_fn(batch):
    waveforms, labels = zip(*batch)

    max_len = max(w.shape[-1] for w in waveforms)
    padded_waveforms = []
    for w in waveforms:
        # Pad to max_len
        if w.shape[-1] < max_len:
            padding = max_len - w.shape[-1]
            w = torch.nn.functional.pad(w, (0, padding))

        padded_waveforms.append(w.squeeze())

    batch_waveforms = torch.stack(padded_waveforms)
    return batch_waveforms, torch.tensor(labels)

In [ ]:
train_loader = DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate_fn, drop_last=True)
dev_loader = DataLoader(dev, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate_fn, drop_last=True)
test_loader = DataLoader(test, batch_size=batch_size, shuffle=False, num_workers=2, collate_fn=collate_fn, drop_last=False)

In [ ]:
class EatingClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = AutoModel.from_pretrained(
            "microsoft/wavlm-base-plus"
        )

        hidden = self.backbone.config.hidden_size  # 768

        self.dropout = nn.Dropout(0.2)

        self.classifier = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 7)
        )

    def forward(self, x):

        out = self.backbone(x).last_hidden_state

        # Mean pooling
        out = out.mean(dim=1)

        out = self.dropout(out)

        return self.classifier(out)

model = EatingClassifier().to(device)

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

In [ ]:
def set_trainable_layers(model, strategy, epoch=None, unfreeze_at_epoch=5, num_unfrozen_layers=4):
    """
    strategy: 'frozen', 'late_unfreeze', 'partial_unfreeze'
    """
    if strategy == "frozen":
        model.backbone.requires_grad_(False)

    elif strategy == "late_unfreeze":
        if epoch < unfreeze_at_epoch:
            model.backbone.requires_grad_(False)
        else:
            model.backbone.requires_grad_(True)

    elif strategy == "partial_unfreeze":
        model.backbone.requires_grad_(False)
        for layer in model.backbone.encoder.layers[-num_unfrozen_layers:]:
            for p in layer.parameters():
                p.requires_grad = True

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [
        {"params": model.backbone.parameters(),"lr": backbone_lr,},
        {"params": model.classifier.parameters(), "lr": classifier_lr,},
    ],
    weight_decay=0.01,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

start_time = time.time() # Start timing the training
history = {}

scaler = torch.amp.GradScaler('cuda')

best_acc = 0
for epoch in range(num_epochs):
    set_trainable_layers(model, strategy, epoch)

    # Training phase
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    all_train_labels = []
    all_train_preds = []

    for batch_audio, batch_labels in train_loader:
        batch_audio = batch_audio.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(batch_audio)
            loss = criterion(outputs, batch_labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        preds = outputs.argmax(1)
        train_correct += (preds == batch_labels).sum().item()
        train_total += batch_labels.size(0)

        all_train_labels.extend(batch_labels.cpu().numpy())
        all_train_preds.extend(preds.cpu().numpy())

    train_acc = 100 * train_correct / train_total
    train_f1_macro = f1_score(all_train_labels, all_train_preds, average='macro')

    # Validation phase
    model.eval()
    dev_loss = 0
    dev_correct = 0
    dev_total = 0
    all_dev_labels = []
    all_dev_preds = []

    with torch.no_grad(): # Disable gradient calculations during validation
        for audio, labels in dev_loader:
            audio = audio.to(device)
            labels = labels.to(device)
            outputs = model(audio)
            loss = criterion(outputs, labels)
            dev_loss += loss.item()
            preds = outputs.argmax(1)
            dev_correct += (preds == labels).sum().item()
            dev_total += labels.size(0)

            all_dev_labels.extend(labels.cpu().numpy())
            all_dev_preds.extend(preds.cpu().numpy())

    dev_acc = 100 * dev_correct / dev_total
    dev_f1_macro = f1_score(all_dev_labels, all_dev_preds, average='macro')

    scheduler.step(dev_acc)

    if dev_acc > best_acc:
        best_acc = dev_acc
        torch.save(model.state_dict(), f"best_model_{strategy}.pt")

    history[epoch+1] = {
        "train_loss": train_loss/len(train_loader),
        "train_acc": train_acc,
        "train_f1_macro": train_f1_macro,
        "dev_loss": dev_loss/len(dev_loader),
        "dev_acc": dev_acc,
        "dev_f1_macro": dev_f1_macro,
    }
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(
        f"Train Loss: {train_loss/len(train_loader):.4f} | "
        f"Train Acc: {train_acc:.2f}% | Train F1 Macro: {train_f1_macro:.4f}"
    )
    print(
        f"Dev Loss: {dev_loss/len(dev_loader):.4f} | "
        f"Dev Acc: {dev_acc:.2f}% | Dev F1 Macro: {dev_f1_macro:.4f}"
    )
    print(f"Best Dev Acc: {best_acc:.2f}%")

end_time = time.time() # End timing the training
training_time = end_time - start_time
print(f"\nTotal Training Time: {training_time:.2f} seconds")

### Evaluation on Test Set

In [ ]:
import json, csv

def save_run_results(strategy, history, y_true, y_pred, class_names, output_dir="results"):
    run_dir = Path(output_dir) / strategy
    run_dir.mkdir(parents=True, exist_ok=True)

    # training log
    with open(run_dir / "training_log.csv", "w", newline="") as f:
        writer = csv.writer(f, delimiter=',')
        writer.writerow(history[1].keys())
        for h in history.values():
            writer.writerow(h.values())

    # classification report
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    with open(run_dir / "classification_report.json", "w") as f:
        json.dump(report, f, indent=2)

    # confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    np.savetxt(run_dir / "confusion_matrix.csv", cm, delimiter=",", fmt="%d")

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix — {strategy}')
    plt.savefig(run_dir / "confusion_matrix.png", bbox_inches="tight")
    plt.close()

    # summary metrics
    test_metrics[strategy] = {
        "strategy": strategy,
        "test_accuracy": accuracy_score(y_true, y_pred),
        "test_f1_macro": f1_score(y_true, y_pred, average="macro"),
        "best_dev_acc": max(h["dev_acc"] for h in history.values()),
        "epochs_trained": len(history),
    }
    with open(Path(output_dir) / "test_metrics.json", "w") as f:
        json.dump(test_metrics, f, indent=2)

In [ ]:
# Load the best model weights
model.load_state_dict(torch.load(f"best_model_{strategy}.pt"))
model.eval() # Set model to evaluation mode

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():
    for batch_audio, batch_labels in test_loader:
        batch_audio = batch_audio.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_audio)

        probabilities = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)

        all_labels.extend(batch_labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())
        all_probabilities.extend(probabilities.cpu().numpy())

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)
all_probabilities = np.array(all_probabilities)

# Map numerical labels back to food names for better readability
food_label_map_inverse = {
    0: "Apple",
    1: "Banana",
    2: "Biscuit",
    3: "Crisp",
    4: "Haribo",
    5: "Nectarine",
    6: "No_Food"
}

class_names = [food_label_map_inverse[i] for i in sorted(food_label_map_inverse.keys())]

In [ ]:
save_run_results(strategy, history, all_labels, all_predictions, class_names, output_dir)

In [ ]:
import shutil
import os

# Create a zip archive of the results directory
zip_path = 'results.zip'
shutil.make_archive('results', 'zip', output_dir)
print(f"'{zip_path}' created successfully.")